In [1]:
import pandas as pd
import json
from tqdm import tqdm
from pyterrier_t5 import MonoT5ReRanker

In [2]:
full_df = pd.read_csv("/nfs/primary/SPLADE/splade_sans/data/training_data_w_qids.csv")
json_file = pd.read_json("hn_splade_sans1_all_sensitive_32_negs_docnos.jsonl", lines = True)

In [3]:
text_lookup = dict(zip(full_df['medline_ui'], full_df['text']))#
query_lookup = dict(zip(full_df['qid'], full_df['best_query']))

In [4]:
create_df = []
for _, row in tqdm(json_file.iterrows(), total = len(json_file), desc = "creating df"):
    qid = row.query
    query = query_lookup[qid]
    pos_docno = row.docno_a
    pos_text = text_lookup[pos_docno]
    negative_docs = row.docno_b
    create_df.append({"qid" : qid, "query" : query, "docno" : pos_docno, "text" : pos_text, "pos" : 1})
    for d in negative_docs:
        negative_docno = int(d)
        negative_text = text_lookup[negative_docno]
        create_df.append({"qid" : qid, "query" : query, "docno" : negative_docno, "text" : negative_text, "pos" : 0})

creating df: 100%|██████████| 334065/334065 [00:09<00:00, 35099.36it/s]


In [5]:
df = pd.DataFrame(create_df)

In [9]:
monot5 = MonoT5ReRanker(verbose = True, batch_size = 64)

In [36]:
scored = monot5(df)

monoT5: 100%|██████████| 26/26 [00:07<00:00,  3.59batches/s]


In [45]:
scores = scored.groupby('qid').apply(lambda x: x.set_index('docno')['score'].to_dict()).to_dict()
with open('scores.json', 'w') as f:
    json.dump(scores, f)

/tmp/ipykernel_1785/2144926164.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  scores = scored.groupby('qid').apply(lambda x: x.set_index('docno')['score'].to_dict()).to_dict()
